# 03 - Feature Engineering and Pre-Processing

##### 03-00 General Model Plan

Feature Engineering

- Create `duration_days` from the cleaned date columns
- Remove invalid and outlier duration rows
- Create the target variable from duration ranges
- Save the final cleaned modeling dataset

After feature engineering (Python training script): 

Pre-Processing

- Split dataset into training and testing subsets
- Split pre-processing steps into 3 groups (Text, Categories, and Words/Lengths)
- Use TF-IDF vectorization to convert text values (summaries and descriptions) to numeric matrices
- Use One Hot Encoder to encode catgories into numeric values with no relations, e.g, Bug and Improvement are completely unrelated
- Use Standard Scaler to standardizes the numeric values

Model training

- Use Logistic Regression
- Train and test model against test subset
- Evaluate and study model metrics


## 03-01 Importing CSV file and defining the target

Before we start training the model using the cleaned dataset, we need to create the target value from the cleaned date columns.

In [55]:
from pathlib import Path
import pandas as pd

jira_df = pd.read_csv("../data/processed/jira_issues_cleaned.csv", index_col=0)

task_df = jira_df.copy()

for col in ["created", "resolutiondate"]:
    task_df[col] = pd.to_datetime(task_df[col])

task_df.head()

,summary,description,priority_name,issuetype_name,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate
0,Update config browser to work with the new syntax,The config browser used Velocity calling the t...,Minor,Improvement,49,9,176,28,2005-01-01 07:47:50,2005-01-01 07:50:46
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,Blocker,Bug,52,11,3445,206,2004-12-25 22:50:30,2004-12-30 05:30:36
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",Critical,Bug,43,8,2560,124,2005-01-01 13:52:46,2005-01-02 15:21:00
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",Major,Improvement,66,7,848,138,2004-12-27 01:34:17,2005-01-03 10:21:28
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,Major,Bug,65,10,419,65,2004-12-28 18:50:52,2005-01-03 11:29:27


## 03-02 Create duration days derived variable

Calculate task duration in days.

This target is derived from:

`duration_days = resolutiondate - created`

In [56]:
task_df["duration_days"] = (task_df["resolutiondate"] - task_df["created"]).dt.total_seconds() / (60 * 60 * 24)

task_df["duration_days"].describe()

count    937217.000000
mean        202.076209
std         518.027076
min           0.000000
25%           1.042118
50%          11.794444
75%         110.378021
max        8001.517257
Name: duration_days, dtype: float64

The above statistics show target-distribution issues that need to be handled before modeling:

- The target has a strong long-tail pattern.
- Extreme outliers pull the mean far above the median.

#### Inspect long-tail durations

Before removing long-running tasks, first inspect how many rows would be affected.

In [57]:
display(task_df["duration_days"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

duration_days_over_90 = (task_df["duration_days"] > 90).sum()
print(f"Rows with duration_days > 90: {duration_days_over_90:,}")

count    937217.000000
mean        202.076209
std         518.027076
min           0.000000
50%          11.794444
75%         110.378021
90%         617.786472
95%        1185.956884
99%        2692.405918
max        8001.517257
Name: duration_days, dtype: float64

Rows with duration_days > 90: 253,255


Removing this many rows is a significant change, although the dataset may still contain enough records for modeling.

Tasks that exceed 90 days may be signs of:

- stale issues
- abandoned tickets
- imported/migrated issues
- tickets closed administratively
- issues left open for months/years without active work
- old backlog cleanup

These rows may not represent normal task-completion behavior, so removing or capping them can make the target easier for the model to learn.

In [58]:
task_df = task_df[task_df["duration_days"] <= 90].copy()

display(task_df.shape)
task_df.describe()

(683962, 11)

,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate,duration_days
count,683962.000000,683962.000000,6.839620e+05,683962.000000,683962,683962,683962.000000
mean,57.161986,7.905357,9.118259e+02,79.719495,2015-11-01 12:18:32.174823,2015-11-14 22:33:03.558047,13.426752
min,0.000000,0.000000,0.000000e+00,0.000000,2002-05-10 18:37:38,2002-05-10 18:54:17,0.000000
25%,40.000000,5.000000,1.170000e+02,16.000000,2012-06-28 23:22:01.250000,2012-07-10 23:10:59.250000,0.450975
50%,54.000000,7.000000,2.870000e+02,40.000000,2016-02-06 10:30:55.500000,2016-02-19 18:24:04,3.870475
75%,70.000000,10.000000,6.780000e+02,84.000000,2019-08-19 15:52:34.500000,2019-09-02 22:51:39,17.171282
max,255.000000,48.000000,7.615811e+06,351933.000000,2024-11-06 14:09:26,2025-01-27 21:10:15,89.999618
std,23.982913,3.658377,1.198356e+04,613.210296,NaN,NaN,20.264542


Tasks resolved in under 4 hours may represent very small fixes, administrative updates, or tickets closed immediately after creation. Removing them can reduce short-duration skew, but this should be treated as a modeling decision rather than a universal rule.

In [59]:
task_df = task_df[task_df["duration_days"] >= (2/24)].copy()

## 03-03 Create duration classification target

Create a classification target from `duration_days`.

The goal is to choose ranges that are:

- simple enough to explain
- not overly skewed toward one class
- aligned with what the task summaries and descriptions appear to represent

The earlier EDA showed two important cleanup decisions before classification:

- remove tasks completed in under 2 hours
- remove very long tasks above the selected long-tail cutoff

After those removals, compare a few possible class ranges before choosing the final classes.

### Duration ranges

- **Short:** same day and up to 3 days
- **Standard:** greater than 3 days and up to 3 weeks
- **Long-term:** greater than 3 weeks


In [62]:
def duration_category(days):
    if days <= 3:
        return "Short"
    if days <= 15:
        return "Standard"
    return "Long-running"

duration_order = ["Short", "Standard", "Long-running"]

task_df["duration_category"] = task_df["duration_days"].apply(duration_category)

class_counts = task_df["duration_category"].value_counts().reindex(duration_order)
class_percentages = (task_df["duration_category"].value_counts(normalize=True).reindex(duration_order).mul(100).round(2))

duration_class_summary = pd.DataFrame({
    "count": class_counts,
    "percent": class_percentages,
})

display(task_df.shape)
display(task_df.describe())

duration_class_summary

(578204, 12)

,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate,duration_days
count,578204.000000,578204.000000,5.782040e+05,578204.000000,578204,578204,578204.000000
mean,57.441600,7.928664,9.652589e+02,84.179705,2015-12-25 03:46:02.245510,2016-01-10 00:51:09.366366,15.878555
min,1.000000,1.000000,0.000000e+00,0.000000,2002-05-10 18:37:38,2002-06-03 18:35:07,0.083333
25%,41.000000,5.000000,1.290000e+02,18.000000,2012-09-01 09:37:30.500000,2012-09-16 18:02:23.250000,1.239546
50%,54.000000,7.000000,3.100000e+02,43.000000,2016-04-28 02:43:28.500000,2016-05-13 00:10:21,6.050874
75%,70.000000,10.000000,7.260000e+02,89.000000,2019-11-06 15:39:18,2019-11-20 21:44:09.750000,21.893226
max,255.000000,48.000000,7.615811e+06,351933.000000,2024-11-06 12:43:08,2025-01-27 21:10:15,89.999618
std,23.970972,3.661370,1.278791e+04,649.192147,NaN,NaN,21.139684


,count,percent
duration_category,,
Short,213274,36.89
Standard,179831,31.10
Long-running,185099,32.01


### 03-03.1 Spot-check task text by class

Sample a few summaries and descriptions from each final class. This is a manual sanity check that the classes are not obviously misleading for the type of task text the model will receive.

In [63]:
sample_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "duration_category",
]

for category in duration_order:
    print(category)
    class_sample = task_df.loc[
        task_df["duration_category"] == category,
        sample_columns,
    ].sample(n=5, random_state=42)
    display(class_sample)

Short


,summary,description,priority_name,issuetype_name,duration_category
568721,certificate subject mismatch,During `$ nvm package` I'm getting this error:...,Minor,Bug,Short
613093,Improve zombie detection in test-patch.sh,"In watching INFRA-15074, I was reminded about ...",Minor,Bug,Short
501763,Hive server 2 fails to start in kerberized env...,"HS2 fails to start on kerberized cluster, for ...",Critical,Bug,Short
338770,Remove Hadoop2 support in IsolatedClientLoader,NaN,Major,Sub-task,Short
1020370,Refactor RemoteLogManagerConfig with AbstractC...,The {{RemoteLogManagerConfig}} class has a ver...,Minor,Improvement,Short


Standard


,summary,description,priority_name,issuetype_name,duration_category
280772,Load for global sort table is failing for null...,NaN,Major,Bug,Standard
934976,width is not measured correctly with textinden...,Steps to reproduce: 1. Launch application Actu...,Major,Bug,Standard
913384,Cannot switch between EJB mode and Session Dir...,Found a small typo that went in with the chang...,Major,Bug,Standard
460051,Avoid allocating new KeyValue and according by...,"in HRegion.append, new KeyValues will be alloc...",Major,Improvement,Standard
813755,[hbase-spark] Support Java api for bulkload,"In JavaHBaseContext, there are java api for bu...",Major,New Feature,Standard


Long-running


,summary,description,priority_name,issuetype_name,duration_category
707811,Build Jackrabbit Oak #2383 failed,No description is provided The build Jackrabbi...,Major,Bug,Long-running
434761,new producer performance and bug improvements,We have seen the following issues with the new...,Major,Bug,Long-running
910763,CXF 3.5.0 Generated WSDL root element,"Hi, I'm migrating an existing soap service fro...",Major,Bug,Long-running
833636,TCK: testInjectFlowMap#testInjectFlowMap: Null...,{code:java} Caused by: java.lang.NullPointerEx...,Major,Bug,Long-running
705445,Revisit HBASE-21207 to make all values fully s...,"- Making values, which contain given units (MB...",Minor,Improvement,Long-running


The samples should not be expected to prove the classes are perfect. They only check that the ranges are reasonable enough for the first text-classification model. Actual performance should still be tested during model training.

In [64]:
category_summary_columns = ["issuetype_name", "priority_name"]
minimum_category_percent = 1.0

removed_rows = True

while removed_rows:
    removed_rows = False

    for column in category_summary_columns:
        column_values = task_df[column].fillna("Missing").astype(str)
        category_percentages = column_values.value_counts(normalize=True).mul(100)
        rare_categories = category_percentages[category_percentages < minimum_category_percent].index

        if len(rare_categories) == 0:
            continue

        row_count_before = len(task_df)
        task_df = task_df.loc[~column_values.isin(rare_categories)].copy()
        removed_count = row_count_before - len(task_df)
        removed_rows = removed_rows or removed_count > 0

        print(f"Removed {removed_count:,} rows from rare {column} categories: {list(rare_categories)}")

for column in category_summary_columns:
    value_summary = (
        task_df[column]
        .fillna("Missing")
        .astype(str)
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )
    value_summary["percent"] = (value_summary["count"] / len(task_df) * 100).round(2)

    print(column)
    display(value_summary)

Removed 12,064 rows from rare issuetype_name categories: ['Dependency upgrade', 'Wish', 'Documentation', 'Question', 'Story', 'Technical task', 'Technical Debt', 'Umbrella', 'Suitable Name Search', 'Epic', 'Dependency', 'Brainstorming', 'Request', 'Temp', 'IT Help', 'TCK Challenge', 'RTC', 'Project', 'New JIRA Project', 'Comment', 'Blog - New Blog Request', 'Github Integration', 'New Git Repo', 'Access', 'Planned Work', 'Proposal', 'Outage', 'Requirement', 'Blogs - New Blog User Account Request']
Removed 11,963 rows from rare priority_name categories: ['Low', 'P2', 'Unknown', 'P3', 'Urgent', 'P1', 'P0', 'P4', 'High', 'Not a Priority']
issuetype_name


,issuetype_name,count,percent
0,Bug,289823,52.30
1,Improvement,133429,24.08
2,Sub-task,61534,11.10
3,Task,37520,6.77
4,New Feature,25113,4.53
5,Test,6758,1.22


priority_name


,priority_name,count,percent
0,Major,377690,68.15
1,Minor,100229,18.09
2,Critical,28157,5.08
3,Blocker,24078,4.34
4,Trivial,16328,2.95
5,Normal,7695,1.39


## 03-04 Save final cleaned dataset

Save the final cleaned dataset after feature engineering.

We will also remove the `duration_days` at this stage to avoid leakage since the model is only going to be trained on the duration ranges for classification.

In [65]:
final_cleaned_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "summary_char_count",
    "summary_word_count",
    "description_char_count",
    "description_word_count",
    "duration_category",
]

final_cleaned_df = task_df[final_cleaned_columns].copy()
final_cleaned_df_sample = final_cleaned_df.sample(n=100, random_state=42)

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

final_cleaned_path = output_dir / "final_cleaned.csv"
final_cleaned_df.to_csv(final_cleaned_path, index=False)

final_cleaned_df_sample_path = output_dir / "final_cleaned_sample.csv"
final_cleaned_df_sample.to_csv(final_cleaned_df_sample_path, index=False)

print(f"Final cleaned modeling rows: {final_cleaned_df.shape[0]:,}")
print(f"Final cleaned modeling columns: {final_cleaned_df.shape[1]:,}")
print(f"Saved final cleaned CSV file to: {final_cleaned_path}")
print(f"Saved final cleaned sample CSV file to: {final_cleaned_df_sample_path}")

final_cleaned_df.head()

Final cleaned modeling rows: 554,177
Final cleaned modeling columns: 9
Saved final cleaned CSV file to: ..\data\processed\final_cleaned.csv
Saved final cleaned sample CSV file to: ..\data\processed\final_cleaned_sample.csv


,summary,description,priority_name,issuetype_name,summary_char_count,summary_word_count,description_char_count,description_word_count,duration_category
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,Blocker,Bug,52,11,3445,206,Standard
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",Critical,Bug,43,8,2560,124,Short
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",Major,Improvement,66,7,848,138,Standard
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,Major,Bug,65,10,419,65,Standard
8,Groovy / Java Mismatch with simple class,The following code compiles and runs as a java...,Major,Bug,40,7,1438,65,Long-running


In [61]:
print(f"Final cleaned modeling rows: {final_cleaned_df_sample.shape[0]:,}")
print(f"Final cleaned modeling columns: {final_cleaned_df_sample.shape[1]:,}")

Final cleaned modeling rows: 100
Final cleaned modeling columns: 9
